[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/regge_calculus/regge_calculus.ipynb)

# Regge Calculus

Regge calculus describes a surface by its edge lengths alone, which determine distances and angles within its triangles. When those angles fail to fill a full turn around an interior vertex, the surface is intrinsically curved. The same lengths also determine how current flows across it.

A symmetric correlation reads squared length through `metric(edge) & edge`. Corner turns carry curvature as bivectors, and the gradient's adjugate brings local current responses back to the vertices. All three constructions live on one fixed reference mesh.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from numga.algebras import VGA2D as ga
from numga.sparse import SparseExtensor, spdiag
from examples import instantiate
from examples.surfaces.regge_calculus import render, scenarios
from examples.util.animation import save_figure

mesh_module = instantiate("examples.util.mesh", ga)
Mesh, as_scalar, mv = mesh_module.Mesh, mesh_module.as_scalar, mesh_module.mv
Scalar = ga.gatype.scalar()                                # Scalar+
Plane = ga.gatype.vector()                                 # Vector+
Point = ga.gatype.antivector()                             # Antivector-

np.set_printoptions(precision=4, suppress=True)

## Edge length correlations

An edge's squared length is a reading of a displacement against a plane. On a triangle, three such readings determine a symmetric correlation. Its basis maps each respond to one edge and vanish on the other two, so each edge length has an independent geometric role.


In [ ]:
n_triangle_subdivisions = 24
edge_stretches = np.array([1.45, 0.65, 1.25])              # [corners]

In [ ]:
corners, points = scenarios.single_triangle(n_triangle_subdivisions)  # [corners] Point-, [samples] Point-
sides = corners[[2, 0, 1]] - corners[[1, 2, 0]]            # [corners] Point-
patch = (sides[0] & sides[1]) / 2                          # [] Patch-
gradients = sides / (2 * patch)                            # [corners] Plane+

# Along the two sides meeting at a corner, one endpoint gradient vanishes while on the opposite
# side the two endpoint gradients pair to -1 and +1, making their symmetric product dual to the squared edge lengths.
first, second = gradients[[1, 2, 0]], gradients[[2, 0, 1]]  # [corners] Plane+, [corners] Plane+
basis = -(first * (second & Point) + second * (first & Point)) / 2  # [corners] Plane+ <- Point-

squared_lengths = sides.norm_squared() * edge_stretches    # [corners] Scalar+
metric = (squared_lengths * basis).sum(axis=0)             # [] Plane+ <- Point-
save_figure(
    render.draw_metric_basis(corners, points, basis, metric),
    "regge_metric_basis",
);

Each basis map reads its own edge (orange) and vanishes along two null lines (dashed) parallel to the other edges, with opposite signs on either side of them. Weighted by the squared lengths, their sum in the fourth panel is the metric, whose equal-length contours (orange) are ellipses against the dashed circles of the reference triangle.

In tensor notation, the Regge basis is $\Psi_{ij}=-\frac12(d\lambda_i\otimes d\lambda_j+d\lambda_j\otimes d\lambda_i)$ and the reconstructed metric is $g_T=\sum_{ij}\ell_{ij}^2\Psi_{ij}$.


The metric also measures the intrinsic triangle's area and its corner angles, each the logarithm of a rotor.

In [ ]:
# The wedge carries the correlation's area scaling; pairing it with the reference patch
# and dividing by two gives the intrinsic area squared.
intrinsic_area = (((metric(sides[0]) ^ metric(sides[1])) & patch) / 2).square_root()  # [] Scalar+

# Combining the metric inner product of the two sides meeting at each corner with twice the intrinsic
# area in the even subalgebra forms a rotor whose norm is the product of those two side lengths.
dots = -(metric(sides[[1, 2, 0]]) & sides[[2, 0, 1]])      # [corners] Scalar+
corner_rotors = (dots + 2 * intrinsic_area * mv.xy).normalized()  # [corners] Rotor+
corner_turns = corner_rotors.log()                         # [corners] Bivector+

In [ ]:
print("Reference vs intrinsic area:", patch.norm().to_array(), intrinsic_area.to_array())
print("Corner turns (radians):", (corner_turns & mv.xy.dual()).to_array())


## Curvature from corner turns

The same reference disk can carry different intrinsic lengths. Lengthening radial motion while shortening circumferential motion leaves too little angle around each interior vertex; reversing that balance leaves too much. The missing or excess turn records curvature. The drawing keeps the reference vertices fixed and shows these intrinsic changes rather than a computed embedding in space.


In [ ]:
n_disk_divisions = 9
disk_radius = 1.0
radial_rates = np.array([0.0, 3.2, -0.45])                  # [cases]
hoop_rates = np.array([0.0, -0.78, 3.2])                    # [cases]

In [ ]:
mesh, vertices = scenarios.disk(disk_radius, n_disk_divisions)  # Mesh, Point-[vertices]
edges = mesh.d[0] * vertices                               # Point-[edges]
centres = mesh.incidence[2, 0].mean(axis=0) * vertices     # Point-[faces]
face_sides = mesh.boundary[2, 1] * edges                   # [corners] Point-[faces]
patches = (face_sides[0] & face_sides[1]) / 2              # Patch-[faces]
areas = patches.norm()                                     # Scalar+[faces]
corner_gradients = face_sides / (2 * patches)              # [corners] Plane+[faces]
first_grad, second_grad = corner_gradients[[1, 2, 0]], corner_gradients[[2, 0, 1]]  # [corners] Plane+[faces], [corners] Plane+[faces]
face_basis = -(first_grad * (second_grad & Point) + second_grad * (first_grad & Point)) / 2  # [corners] (Plane+ <- Point-)[faces]

In [ ]:
midpoints = mesh.incidence[1, 0].mean(axis=0) * vertices   # Point-[edges]
# The dot reading measures radial motion times radius; the join measures swept area,
# hence circumferential motion times radius. Their squares choose the two length responses.
edge_squared_lengths = (
    edges.norm_squared()
    + radial_rates * midpoints.dot(edges).squared()
    + hoop_rates * (midpoints & edges).norm_squared()
)                                                          # [cases] Scalar+[edges]

metrics = (spdiag(face_basis) * mesh.incidence[2, 1]).sum(axis=0) * edge_squared_lengths  # [cases] (Plane+ <- Point-)[faces]
intrinsic_areas = (((metrics(face_sides[0]) ^ metrics(face_sides[1])) & patches) / 2).square_root()  # [cases] Scalar+[faces]
face_dots = -(metrics[:, None](face_sides[[1, 2, 0]]) & face_sides[[2, 0, 1]])  # [cases, corners] Scalar+[faces]
face_rotors = (face_dots + 2 * intrinsic_areas[:, None] * mv.xy).normalized()  # [cases, corners] Rotor+[faces]
face_turns = face_rotors.log()                             # [cases, corners] Bivector+[faces]

On the boundary a flat surface turns through a half turn rather than a full one. The defect at each vertex, divided by the vertex's third of the intrinsic area around it, is the Gaussian curvature as a bivector.

In [ ]:
# Reversing the corner-to-vertex incidence map both sums over the corners around each vertex and reverses
# each corner's bivector turn, subtracting the incident corner turns from the flat target turn.
unit_faces = as_scalar(np.ones(len(mesh.faces))).field()   # Scalar+[faces]
corner_mean = mesh.incidence[2, 0].mean(axis=0)            # Scalar+[faces <- vertices]
vertex_areas = ~corner_mean * intrinsic_areas              # [cases] Scalar+[vertices]
boundary = np.unique(mesh.edges[mesh.boundary_edges])
is_boundary = np.isin(np.arange(mesh.n_vertices), boundary)
target_turns = as_scalar(np.where(is_boundary, np.pi, 2 * np.pi)).field() * mv.xy  # Bivector+[vertices]
defects = target_turns + ~(spdiag(face_turns) * mesh.incidence[2, 0]).sum(axis=-1) * unit_faces  # [cases] Bivector+[vertices]
curvatures = defects / vertex_areas                        # [cases] Bivector+[vertices]

save_figure(
    render.draw_curvature_defects(mesh, vertices, centres, metrics, curvatures),
    "regge_curvature_defects",
);

Lengthening radial distances while shortening circumferential ones curves the middle disk positively (orange); lengthening circumferential distances curves the right disk negatively (blue); the flat disk has no interior curvature. Each ellipse holds the displacements up to one common intrinsic length against a dashed reference circle, red where radial stretch dominates and blue where circumferential stretch does.

In the angle-defect notation of Regge calculus, interior defects $\kappa_v=2\pi-\sum_{f\ni v}\theta_{f,v}$ and boundary turning angles $\kappa_v^\partial=\pi-\sum_{f\ni v}\theta_{f,v}$ satisfy $\sum_{v\notin\partial M}\kappa_v+\sum_{v\in\partial M}\kappa_v^\partial=2\pi\chi(M)$.


In [ ]:
print("Gaussian curvature at centre (flat, radial stretch, hoop stretch):", (curvatures.batch()[:, 0] & mv.xy.dual()).to_array())


## Current on the intrinsic surface

A potential gradient reads changes along displacements; conductivity maps that gradient to the current it drives. Here each triangle conducts equally in every intrinsic direction, although its current can look anisotropic in the reference drawing. The adjugate balances those local responses at the vertices, giving the steady flow between two electrodes.


In [ ]:
electrode_threshold = 0.78

In [ ]:
# Each side dyad turns a gradient reading into flow along that side;
# the corner cotangents weight the response of the intrinsic triangle.
cotangents = face_dots / (2 * intrinsic_areas[:, None])    # [cases, corners] Scalar+[faces]
conductivities = ((cotangents / 2) * (face_sides * (Plane & face_sides))).sum(axis=-1)  # [cases] (Point- <- Plane+)[faces]

gradient = (spdiag(corner_gradients) * mesh.incidence[2, 0]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- vertices]
centroid_forms = (corner_gradients[[2, 0, 1]] - corner_gradients[[1, 2, 0]]) / 3  # [corners] Plane+[faces]
whitney1 = (spdiag(centroid_forms) * mesh.boundary[2, 1]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- edges]

In [ ]:
# The adjugate carries each triangle's current response back to its vertex potentials.
conductance = gradient.adjugate()(spdiag(conductivities)(gradient))  # [cases] (Amount- <- Scalar+)[vertices <- vertices]
abscissa = (mv.x & vertices).batch()                       # [vertices] Scalar+
electrodes = boundary[abscissa[boundary].norm() > electrode_threshold]
pinned = SparseExtensor.selection(mv.context, electrodes, mesh.n_vertices)
prescribed = ~pinned * (pinned * (mv.x & vertices))        # Scalar+[vertices]
free_vertices = np.setdiff1d(np.arange(mesh.n_vertices), electrodes)
free = ~SparseExtensor.selection(mv.context, free_vertices, mesh.n_vertices) * Scalar  # (Scalar+ <- Scalar+)[vertices <- free]
rhs = -free.adjugate()(conductance(prescribed))            # [cases] Amount-[free]
potential = prescribed + free(free.adjugate()(conductance(free)).solve(rhs))  # [cases] Scalar+[vertices]
integrated_flux = -conductivities(gradient(potential))     # [cases] Point-[faces]
current = integrated_flux / areas                          # [cases] Point-[faces]
edge_flux = whitney1.adjugate()(integrated_flux)           # [cases] Amount-[edges]

The drawing traces the current along the level curves of its streamfunction.

In [ ]:
# Because the area-integrated flux is divergence-free on the reference mesh, turning it by a quarter turn
# gives the area-weighted reference gradient of the streamfunction whose level curves follow the intrinsic flow.
quarter_turn = (mv.xy * (-np.pi / 4)).exp()                # [] Rotor+
laplacian = gradient.adjugate()(spdiag(areas * Plane.dual())(gradient))  # (Amount- <- Scalar+)[vertices <- vertices]
unpinned = ~SparseExtensor.selection(mv.context, np.arange(1, mesh.n_vertices), mesh.n_vertices) * Scalar  # (Scalar+ <- Scalar+)[vertices <- unpinned]
stream_rhs = unpinned.adjugate()(gradient.adjugate()(quarter_turn >> integrated_flux))  # [cases] Amount-[unpinned]
streamfunction = unpinned(unpinned.adjugate()(laplacian(unpinned)).solve(stream_rhs))  # [cases] Scalar+[vertices]

save_figure(
    render.draw_intrinsic_flow(mesh, vertices, centres, potential, streamfunction, current),
    "regge_intrinsic_laplacian",
);

Shorter circumferential distances in the middle disk favour paths near the rim; longer ones in the right disk concentrate the current through the centre. Colour and dashed contours show the potential, solid curves follow the current from the right electrode to the left, and arrows show the flux density in reference coordinates, scaled per panel.

In tensor notation, the area-weighted inverse metric is $A_Tg_T^{-1}=\frac12\sum_k\cot\theta_k\,s_k\otimes s_k$. In finite element notation, its pullback through the corner gradients is the intrinsic cotangent stiffness matrix.


In [ ]:
origin_face = centres.norm().batch().argmin()
print("Reference current magnitude at centre (flat, radial stretch, hoop stretch):", current.batch()[:, origin_face].norm().to_array())


In [ ]:
# checks
side_readings = basis[:, None](sides[None, :]) & sides[None, :]
np.testing.assert_allclose(side_readings.kernel, np.eye(3)[..., None], atol=1e-14)
np.testing.assert_allclose(((metric(sides) & sides) - squared_lengths).kernel, 0.0, atol=1e-14)

euclidean_metric = (sides.norm_squared() * basis).sum(axis=0)
np.testing.assert_allclose((euclidean_metric - Point.dual_inverse()).kernel, 0.0, atol=1e-14)

lengths = np.sqrt(squared_lengths.kernel[..., 0])
semiperimeter = lengths.sum() / 2
heron_area = np.sqrt(semiperimeter * np.prod(semiperimeter - lengths))
np.testing.assert_allclose(intrinsic_area.kernel, heron_area, atol=1e-14)
np.testing.assert_allclose(corner_rotors.scalar_norm_squared().kernel, 1.0, atol=1e-14)
np.testing.assert_allclose((corner_turns.sum(axis=0) - np.pi * mv.xy).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose((corner_rotors[0] * corner_rotors[1] * corner_rotors[2]).kernel, [-1.0, 0.0], atol=1e-14)

np.testing.assert_allclose((defects.sites.sum() - 2 * np.pi * mv.xy).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose(defects.batch()[0, ~is_boundary].kernel, 0.0, atol=1e-13)
assert (curvatures.batch()[1, 0] & mv.xy.dual()) > 0
assert (curvatures.batch()[2, 0] & mv.xy.dual()) < 0

inverse_conductivities = intrinsic_areas * metrics.batch().inverse().field()
np.testing.assert_allclose((conductivities - inverse_conductivities).kernel, 0.0, atol=1e-13)
np.testing.assert_allclose((conductance - conductance.adjugate()).cells.kernel, 0.0, atol=1e-13)
np.testing.assert_allclose(free.adjugate()(conductance(potential)).kernel, 0.0, atol=1e-11)